# Autoencoding and Self-Supervision

In [1]:
# Import dependencies
import torch
import torchvision
from torchvision import transforms
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import *
from sklearn.preprocessing import StandardScaler
import numpy as np
import matplotlib.pyplot as plt
#from idlmam import *
from sklearn.metrics import accuracy_score

In [2]:
# Import the dl_utils.py file
import sys, os

# Add the project root to the python path
sys.path.insert(0, os.path.abspath("../DL_utils_EZA")) # Goes up one level from current folder

# Autoreload: picks up edits to dl_utils.py without restarting the kernel
%load_ext autoreload
%autoreload 2

# Import my own utilities
from dl_utils import * #import all my functions

print(" dl_utils loaded")

# Verification of CUDA present and force deterministic bahaviour for reproducibility with forced GPU usage

torch.backends.cudnn.benchmark = False  # False is better for RNNs with variable input sizes, True for CNNs

if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'Device available: {device}')

 dl_utils loaded
Device available: cuda


/home/efrain/Documents/Bio-AI-foundations/DL_utils_EZA/dl_utils.py:9: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


## Implementing a *Principal Component Analysis* (PCA) as an Autoencoder
We define the network function: $$f(x) = xWW^{T}$$

In [3]:
# Define constants for features
D = 28*28     # Values in the input
n = 2         # Hidden layers size 
C = 1         # Number of channles
classes = 10  # Number of classes

In [4]:
# View class to conver the image shape
class View(nn.Module):
    def __init__(self, *shape):
        super(View, self).__init__()
        self.shape = shape
    def forward(self, input):
        return input.view(*self.shape) 

In [5]:
# Define the TransposeLinear class
class TransposeLinear(nn.Module):   # Class extens nn.Module and all PyTorch layers extend this
    def __init__(self, linearLayer, bias=True):
        """ 
        LinearLayer: is the layer that we want to use the transpose of to produce the ouput of this layer. 
                     The linea layer represents W, and this layer represents W^T.
        bias: if True, creates a new bias term b that is learned separately from what is in linearLayer. If
              false no bias vector is used.
        """

        super().__init__()
        self.weight = linearLayer.weight     # Creates the new variable weight to store a reference to original weight
        if bias:
            self.bias = nn.Parameter(torch.Tensor(linearLayer.weight.shape[1])) # Creates new bias vector
        else:
            self.register_parameter('bias', None)
        
    def forward(self, x):
                return F.linear(x, self.weight.t(), self.bias)
    

In [6]:
# Implement PCA

# Define the linear layer
linearLayer = nn.Linear(D, n, bias=False)

# Encoder flattens and use the linear layer
pca_encoder = nn.Sequential(
    nn.Flatten(),
    linearLayer
    )

    # Decoder uses the TransposeLienar layer.
pca_decoder = nn.Sequential(
    TransposeLinear(linearLayer, bias=False), 
    View(-1, 1, 28, 28)   # Shapes data back to original form
    )

    # Defineas a final PCA model with encoder, decoder
pca_model = nn.Sequential(
    pca_encoder,
    pca_decoder
    )

    # Add the constraint
nn.init.orthogonal_(linearLayer.weight)

    #Original Loss function
mse_loss = nn.MSELoss()

In [7]:
# The PCA loss function
def mseWithOrthoLoss(x, y):
    W = linearLayer.weight   
    I = torch.eye(W.shape[0]).to(device) # Identity matrix target for regularization

    normal_loss = mse_loss(x, y)  # Computes original loss
    regularization_loss = 0.1*mse_loss(torch.mm(W, W.t()), I) # computes regularizer penalty
          
    return normal_loss + regularization_loss 

### Implementing PCA with PyTorch
Crate a Wrapper for the MNIST dataset. It will transform (x, y) to (x , x)

In [8]:
# Add the class AtoEncodeDataset
class AutoEncodeDataset(Dataset):
    """  
    Takes a dataset with (x, y) and convets to (x, x)
    """

    def __init__(self, dataset):
        self.dataset = dataset
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        x, y = self.dataset.__getitem__(idx)
        return x, x

In [9]:
# Load the MNIST dataset and transform it into (x, x)
train_data = (AutoEncodeDataset(torchvision.datasets.MNIST("./data/Autoencoder", train=True, 
                                                           transform=transforms.ToTensor(), 
                                                           download=True))
)

test_data_xy = torchvision.datasets.MNIST("./data/Autoencoder", train=True,
                                          transform=transforms.ToTensor(),
                                          download=True
)

test_data_xx = AutoEncodeDataset(test_data_xy)



In [10]:
# Data loaders
train_loader = DataLoader(train_data, 
                          batch_size=128, 
                          shuffle=True,
                          num_workers=4,
                          pin_memory=True,
                          persistent_workers=True,
                          drop_last=True
)

test_loader = DataLoader(test_data_xx, 
                        batch_size=128*2,
                        shuffle=False,
                        num_workers=4,
                        persistent_workers=True,
                        pin_memory=True
)

In [11]:
# Train the network
train_network_EZA(pca_model,
                  mseWithOrthoLoss,
                  train_loader,
                  test_loader=test_loader,
                  epochs=10,
                  device=device
)

Epoch: 100%|██████████| 10/10 [00:55<00:00,  5.59s/it]


,epoch,total time,train loss,test loss
0,0,2.690467,0.063534,0.057990
1,1,2.295093,0.057902,0.057877
2,2,2.739932,0.057886,0.057865
3,3,3.097925,0.057888,0.057905
4,4,3.109265,0.057887,0.057878
5,5,3.099641,0.057890,0.057888
6,6,3.031635,0.057883,0.057894
7,7,3.044601,0.057893,0.057891
8,8,3.049503,0.057891,0.057878
9,9,3.082907,0.057892,0.057881


### Visualizing PCA results